# Step 5 — Baseline Cleaning with Pandas

Cleans `raw_data.csv` produced by the crawler. Each cell is one cleaning step so the pipeline is easy to read and audit. The final cell saves `clean_data.csv` and a callable `clean_pandas` lives in `cleaners.py` for reuse in step 6/7.

In [15]:
import pandas as pd
from cleaners import clean_pandas, parse_price, parse_relative

RAW = 'raw_data.csv'
OUT = 'clean_data.csv'

## 1. Load

In [16]:
df = pd.read_csv(RAW)
print('shape:', df.shape)
df.head()

shape: (200033, 13)


C:\Users\user\AppData\Local\Temp\ipykernel_19796\4087801202.py:1: DtypeWarning: Columns (0: likes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW)


,listing_id,product,price,original_price,condition,seller,time_posted,likes,listing_url,source_url,category,buyer_protection,delivery_info
0,1437686064,Gaming PC RTX 5060Ti + Ryzen 7 3700X | 16GB RAM,"RM3,000",NaN,Lightly used,electroparts,2 hours ago,1.0,https://www.carousell.com.my/p/gaming-pc-rtx-5...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,NaN
1,1437681886,PC Gaming Ryzen 5 5500 RTX 4060 1TB SSD,"RM2,550",NaN,Like new,nadia_marissa,3 hours ago,3.0,https://www.carousell.com.my/p/pc-gaming-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,NaN
2,1431417654,HP All-in-One Desktop PC i3,RM500,NaN,Lightly used,easy4you2016,1 month ago,27.0,https://www.carousell.com.my/p/hp-all-in-one-d...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,NaN
3,1436800248,Gaming PC Ryzen 9 3900X,"RM2,400",NaN,Lightly used,mr_raii,5 days ago,14.0,https://www.carousell.com.my/p/gaming-pc-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,NaN
4,1437391911,PC Gaming Ryzen 5 3600 RTX3060 (USED) - Forza ...,"RM2,000",NaN,Lightly used,zulakmalhakim71,2 days ago,7.0,https://www.carousell.com.my/p/pc-gaming-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,NaN


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200033 entries, 0 to 200032
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   listing_id        200033 non-null  int64 
 1   product           200032 non-null  str   
 2   price             198430 non-null  str   
 3   original_price    22684 non-null   str   
 4   condition         196714 non-null  str   
 5   seller            200033 non-null  str   
 6   time_posted       200033 non-null  str   
 7   likes             161364 non-null  object
 8   listing_url       200033 non-null  str   
 9   source_url        200033 non-null  str   
 10  category          200033 non-null  str   
 11  buyer_protection  200033 non-null  bool  
 12  delivery_info     20251 non-null   str   
dtypes: bool(1), int64(1), object(1), str(10)
memory usage: 18.5+ MB


## 2. Inspect quality

In [18]:
print('Null counts:')
print(df.isna().sum())
print('\nDuplicate listing_id:', df['listing_id'].duplicated().sum())

Null counts:
listing_id               0
product                  1
price                 1603
original_price      177349
condition             3319
seller                   0
time_posted              0
likes                38669
listing_url              0
source_url               0
category                 0
buyer_protection         0
delivery_info       179782
dtype: int64

Duplicate listing_id: 0


## 3. Drop duplicates

In [19]:
before = len(df)
df = df.drop_duplicates(subset=['listing_id'], keep='first')
print(f'Removed {before - len(df)} duplicate rows. Remaining: {len(df)}')

Removed 0 duplicate rows. Remaining: 200033


## 5. Standardize prices
`RM3,000` -> `3000.0`.

In [20]:
df['price'] = df['price'].map(parse_price)
df[['product', 'price']].head()

,product,price
0,Gaming PC RTX 5060Ti + Ryzen 7 3700X | 16GB RAM,3000.0
1,PC Gaming Ryzen 5 5500 RTX 4060 1TB SSD,2550.0
2,HP All-in-One Desktop PC i3,500.0
3,Gaming PC Ryzen 9 3900X,2400.0
4,PC Gaming Ryzen 5 3600 RTX3060 (USED) - Forza ...,2000.0


## 6. Handle missing values

In [21]:
df = df.drop(columns=['original_price'], errors='ignore')

df['likes'] = pd.to_numeric(df['likes'], errors='coerce').fillna(0).astype('Int64')
df['condition'] = df['condition'].fillna('unknown')
df['delivery_info'] = df['delivery_info'].fillna('No deliver info')
df['buyer_protection'] = df['buyer_protection'].fillna('unknown').astype(str)

before = len(df)
df = df.dropna(subset=['price', 'product'])
print(f'Dropped {before - len(df)} rows missing price or product')

Dropped 1604 rows missing price or product


## 7. Normalize text

In [22]:
for col in ('product', 'seller', 'condition', 'category'):
    df[col] = df[col].astype(str).str.strip()
df['condition'] = df['condition'].str.lower()
df['condition'].value_counts().head()

condition
brand new       63925
like new        60968
lightly used    34909
well used       14916
new             14761
Name: count, dtype: int64

## 8. Parse relative timestamps
`'2 hours ago'` -> absolute `posted_at` (datetime).

In [23]:
now = pd.Timestamp.now()
df['posted_at'] = df['time_posted'].map(lambda v: parse_relative(v, now))
df[['time_posted', 'posted_at']].head()

,time_posted,posted_at
0,2 hours ago,2026-05-21 12:46:22.579328
1,3 hours ago,2026-05-21 11:46:22.579328
2,1 month ago,2026-04-21 14:46:22.579328
3,5 days ago,2026-05-16 14:46:22.579328
4,2 days ago,2026-05-19 14:46:22.579328


## 9. Validate dtypes

In [24]:
df['price'] = df['price'].astype(float)
df['posted_at'] = pd.to_datetime(df['posted_at'], errors='coerce')
df = df.reset_index(drop=True)
print(df.dtypes)

listing_id                   int64
product                        str
price                      float64
condition                      str
seller                         str
time_posted                    str
likes                        Int64
listing_url                    str
source_url                     str
category                       str
buyer_protection               str
delivery_info                  str
posted_at           datetime64[us]
dtype: object


## 10. Save

In [25]:
df.to_csv(OUT, index=False)
print(f'Saved {len(df)} rows -> {OUT}')
df.head()

Saved 198429 rows -> clean_data.csv


,listing_id,product,price,condition,seller,time_posted,likes,listing_url,source_url,category,buyer_protection,delivery_info,posted_at
0,1437686064,Gaming PC RTX 5060Ti + Ryzen 7 3700X | 16GB RAM,3000.0,lightly used,electroparts,2 hours ago,1,https://www.carousell.com.my/p/gaming-pc-rtx-5...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-21 12:46:22.579328
1,1437681886,PC Gaming Ryzen 5 5500 RTX 4060 1TB SSD,2550.0,like new,nadia_marissa,3 hours ago,3,https://www.carousell.com.my/p/pc-gaming-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-21 11:46:22.579328
2,1431417654,HP All-in-One Desktop PC i3,500.0,lightly used,easy4you2016,1 month ago,27,https://www.carousell.com.my/p/hp-all-in-one-d...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-04-21 14:46:22.579328
3,1436800248,Gaming PC Ryzen 9 3900X,2400.0,lightly used,mr_raii,5 days ago,14,https://www.carousell.com.my/p/gaming-pc-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-16 14:46:22.579328
4,1437391911,PC Gaming Ryzen 5 3600 RTX3060 (USED) - Forza ...,2000.0,lightly used,zulakmalhakim71,2 days ago,7,https://www.carousell.com.my/p/pc-gaming-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-19 14:46:22.579328


## Reusable function
The same pipeline above lives in `cleaners.clean_pandas(df)` for step 6/7 benchmarks.

In [26]:
import importlib, cleaners
importlib.reload(cleaners)
from cleaners import clean_pandas

raw = pd.read_csv(RAW)
cleaned = clean_pandas(raw)
print('clean_pandas() rows:', len(cleaned))
print('columns:', cleaned.columns.tolist())
assert cleaned['listing_id'].duplicated().sum() == 0
assert cleaned['price'].isna().sum() == 0
assert cleaned['product'].isna().sum() == 0
assert 'original_price' not in cleaned.columns
assert 'delivery_info' not in cleaned.columns
print('Validation passed.')

C:\Users\user\AppData\Local\Temp\ipykernel_19796\1923702949.py:5: DtypeWarning: Columns (0: likes) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(RAW)


clean_pandas() rows: 198429
columns: ['listing_id', 'product', 'price', 'condition', 'seller', 'time_posted', 'likes', 'listing_url', 'source_url', 'category', 'buyer_protection', 'delivery_info', 'posted_at']


AssertionError: 